In [1]:
%pip install faiss-cpu -U sentence-transformers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.3/571.3 kB 19.4 MB/s eta 0:00:00
  Attempting uninstall: sentence-transformers
    Found existing installation: sentence-transformers 5.4.0
    Uninstalling sentence-transformers-5.4.0:
      Successfully uninstalled sentence-transformers-5.4.0




## Read file


In [2]:
def extract_text_from_txt(file_path):
    # Try different encodings
    encodings = ['utf-8', 'latin-1', 'cp1252', 'iso-8859-1']

    for encoding in encodings:
        try:
            with open(file_path, 'r', encoding=encoding) as f:
                return f.read()
        except UnicodeDecodeError:
            continue

    # If all encodings fail, read as binary and decode with error handling
    with open(file_path, 'rb') as f:
        return f.read().decode('utf-8', errors='replace')

In [11]:
all_text = extract_text_from_txt('/content/Box Office Manual (2025) - working.txt')

In [12]:
print(all_text[:100])
print(len(all_text))






BOX OFFICE MANUAL
 ~2026~

VENUE INFORMATION 

Melbourne Town Hall (MTH)

Location: 90/130 Swan
111636


## chunking


In [ ]:
chunk_size = 5000
chunks = [all_text[i : i + chunk_size] for i in range(0, len(all_text), chunk_size)]
len(chunks)

39

In [13]:
from sentence_transformers import SentenceTransformer

In [14]:
# Semantic Chunking Implementation
# This approach uses embeddings to group semantically similar content together

from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.cluster import KMeans
import re

# Load the embedding model for semantic chunking
model_id = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(model_id)

def split_into_sentences(text):
    """Split text into sentences using regex."""
    # Split on common sentence endings
    sentences = re.split(r'(?<=[.!?])\s+', text)
    # Filter out empty sentences
    sentences = [s.strip() for s in sentences if s.strip()]
    return sentences

def semantic_chunking(text, model, max_chunk_size=10, min_chunk_size=2):
    """
    Semantic chunking strategy:
    1. Split text into sentences
    2. Embed each sentence
    3. Use KMeans clustering to group semantically similar sentences
    4. Form chunks from clusters
    """
    # Split text into sentences
    sentences = split_into_sentences(text)
    print(f"Total sentences: {len(sentences)}")

    if len(sentences) < min_chunk_size:
        return [text]

    # Embed each sentence
    sentence_embeddings = model.encode(sentences)

    # Determine number of clusters (chunks)
    # Aim for roughly max_chunk_size sentences per chunk
    n_clusters = max(1, len(sentences) // max_chunk_size)
    n_clusters = min(n_clusters, len(sentences) // min_chunk_size)

    if n_clusters < 1:
        n_clusters = 1

    print(f"Number of clusters: {n_clusters}")

    # Cluster sentences
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(sentence_embeddings)

    # Group sentences by cluster
    clusters = {}
    for i, label in enumerate(cluster_labels):
        if label not in clusters:
            clusters[label] = []
        clusters[label].append(sentences[i])

    # Sort clusters by position in original text (using first sentence index)
    cluster_order = {}
    for label in clusters:
        # Find the first occurrence of any sentence from this cluster
        first_idx = next(i for i, l in enumerate(cluster_labels) if l == label)
        cluster_order[label] = first_idx

    sorted_clusters = sorted(cluster_order.items(), key=lambda x: x[1])

    # Form chunks from clusters
    chunks = []
    for label, _ in sorted_clusters:
        chunk = " ".join(clusters[label])
        chunks.append(chunk)

    return chunks

# Apply semantic chunking
semantic_chunks = semantic_chunking(all_text, model, max_chunk_size=15, min_chunk_size=3)
print(f"Number of semantic chunks: {len(semantic_chunks)}")
print(f"Chunk sizes (chars): {[len(c) for c in semantic_chunks]}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Total sentences: 1219
Number of clusters: 81
Number of semantic chunks: 81
Chunk sizes (chars): [3579, 913, 1012, 3408, 1124, 2568, 1441, 2385, 203, 605, 1374, 29, 5842, 513, 593, 1598, 1412, 1055, 1348, 2121, 550, 2495, 1541, 1058, 874, 1327, 610, 107, 1335, 2140, 698, 542, 5939, 1646, 855, 534, 2356, 1158, 1600, 1330, 1629, 495, 2457, 965, 1871, 1582, 2444, 1784, 1714, 233, 883, 743, 158, 118, 1629, 2135, 2151, 674, 2942, 230, 530, 871, 484, 3660, 137, 3729, 2507, 910, 935, 962, 719, 786, 785, 2080, 300, 305, 130, 264, 152, 640, 1541]


In [17]:
import numpy as np

# Convert semantic_chunks to a numpy array if it's not already
# Assuming semantic_chunks is a list of strings, it can be directly saved.
# If it were a list of embeddings, we would typically concatenate them into a 2D array.
# For a list of strings, np.save will handle it as an object array.
np.save('semantic_chunks.npy', semantic_chunks)

print("Semantic chunks exported to semantic_chunks.npy")

Semantic chunks exported to semantic_chunks.npy


## embed


In [18]:
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')

In [19]:
#call embedding model from Huggingface API
from huggingface_hub import InferenceClient
import os

def embed(text):
    client = InferenceClient(model="sentence-transformers/all-MiniLM-L6-v2", token=hf_token)
    output = client.feature_extraction(text)
    return output

embeddings = embed(semantic_chunks)


In [23]:
embeddings

array([[ 0.04612022, -0.04432139, -0.02112881, ..., -0.02199615,
        -0.1287705 ,  0.06869203],
       [ 0.12571082, -0.0210558 , -0.02605137, ...,  0.0206226 ,
        -0.09175212,  0.05186731],
       [ 0.04184598, -0.05697138, -0.00342095, ...,  0.08735035,
        -0.09457472, -0.01654101],
       ...,
       [-0.03380329,  0.03856072, -0.02104671, ...,  0.05824368,
        -0.02632163,  0.05827608],
       [-0.00076197, -0.13449137,  0.08502165, ...,  0.0354326 ,
        -0.05061976, -0.04880114],
       [-0.05835042,  0.00459803, -0.00041247, ...,  0.05017935,
        -0.00150406, -0.0723778 ]], dtype=float32)

In [24]:
np.save('semantic_embeddings.npy', embeddings)

## Storeing embedding vectors for retrieval


In [ ]:
import faiss

d = embeddings.shape[1]
index = faiss.IndexFlatL2(d)
index.add(embeddings)

when the user asks a question, it is also embedded using the function from above.


## retrieval


In [ ]:
import numpy as np

question = "Where is Melbourne Town Hall?"
question_embeddings = np.array([embed(question)])

D, I = index.search(question_embeddings, k=2)  # distance, index
retrieved_chunk = [semantic_chunks[i] for i in I.tolist()[0]]

we create a prompt template that combines the retrieved chunk that contains the most relevant/simalar info and then the users question


## Chat model


In [ ]:
from mistralai.client import MistralClient
from mistralai import Mistral

client = Mistral(api_key="API_KEY")

def run_mistral(user_message, model="mistral-medium"):
    messages = [{"role":"user", "content":user_message}]
    chat_response = client.chat.complete(model=model, messages=messages)
    return chat_response.choices[0].message.content


run_mistral(prompt)

'Melbourne Town Hall is located at 90/130 Swanston St, Melbourne VIC 3000.'

In [ ]:
from faiss import IndexFlatL2

prompt = """
Context information is below.
---------------------
{context}
---------------------
Given the context information and not prior knowledge, answer the query.
Query: {query}
Answer:
"""


def ask(query: str, index: IndexFlatL2, chunks):
    embedding = embed(query)
    embedding = np.array([embedding])

    _, indexes = index.search(embedding, k=2)
    context = [semantic_chunks[i] for i in indexes.tolist()[0]]

    user_message = prompt.format(context=context, query=query)

    messages = [{"role":"user", "content":user_message}]
    chat_response = client.chat.complete(model="mistral-medium", messages=messages)
    return chat_response.choices[0].message.content


ask("What do i need to bring to Plenary for a venue shift?", index, chunks)

'The context information provided does not mention anything about "Plenary" or what needs to be brought for a venue shift there. The details cover venues like Melbourne Town Hall (MTH), Hamer Hall, and Sidney Myer Music Bowl (SMMB), but there is no reference to a venue named "Plenary."\n\nIf you are referring to a different venue or if "Plenary" is a typo or alternative name for one of the mentioned venues, please clarify or provide additional context. Otherwise, I cannot provide an answer based on the given information.'

In [ ]:
ask("Are there steps in Hamer Hall Stalls?", index, chunks)

'The context information does not explicitly mention the presence of steps in the Stalls section of Hamer Hall. However, it does provide details about the location of Swing Arm Seats in the Stalls section, which are located on the aisle seats in specific rows (B, E, H, L, P, T, and W). This suggests that there are aisles and possibly steps between rows, but it is not explicitly confirmed.\n\nFor more detailed information, you might need to refer to the seat map or additional resources provided in the context, such as the link to the Hamer Hall seat map.'